# 02 - 优化器（Optimizers）

## 学习目标

1. 理解优化器的基类和 5 个核心方法
2. 掌握 SGD 的更新公式、动量、权重衰减
3. 理解 Adam 的自适应学习率机制
4. 理解 AdamW 的解耦权重衰减
5. 掌握参数组和优化器状态的保存/加载

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

## 1. 优化器基类（Optimizer）

所有优化器都继承自 `torch.optim.Optimizer`，提供 5 个核心方法：

| 方法 | 作用 |
|------|------|
| `zero_grad()` | 清零所有参数的梯度 |
| `step()` | 根据梯度更新参数 |
| `add_param_group()` | 添加新的参数组 |
| `state_dict()` | 返回优化器状态字典 |
| `load_state_dict()` | 加载优化器状态字典 |

### 标准训练循环

```python
for data, target in dataloader:
    optimizer.zero_grad()        # 1. 清零梯度
    output = model(data)         # 2. 前向传播
    loss = loss_fn(output, target) # 3. 计算损失
    loss.backward()              # 4. 反向传播
    optimizer.step()             # 5. 更新参数
```

In [ ]:
# 创建一个简单模型用于演示
model = nn.Linear(3, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 查看优化器管理的参数
print("优化器的参数组:")
for i, group in enumerate(optimizer.param_groups):
    print(f"  组 {i}: lr={group['lr']}, params 数量={len(group['params'])}")

In [ ]:
# 5 个核心方法演示
model = nn.Linear(3, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 模拟一个训练步骤
x = torch.randn(2, 3)
target = torch.randn(2, 1)

# 记录更新前的参数
weight_before = model.weight.data.clone()

# 前向 + 反向
output = model(x)
loss = nn.MSELoss()(output, target)
loss.backward()

print(f"梯度: {model.weight.grad}")

# step(): 更新参数
optimizer.step()
print(f"\n更新前权重: {weight_before}")
print(f"更新后权重: {model.weight.data}")
print(f"差值 = -lr * grad: {model.weight.data - weight_before}")
print(f"验证: {-0.01 * model.weight.grad}")

# zero_grad(): 清零梯度
print(f"\n清零前梯度: {model.weight.grad}")
optimizer.zero_grad()
print(f"清零后梯度: {model.weight.grad}")

## 2. SGD（随机梯度下降）

最基本的优化器。

### 基本更新公式
$$\theta_{t+1} = \theta_t - \text{lr} \cdot g_t$$

### 带动量
$$v_t = \mu \cdot v_{t-1} + g_t$$
$$\theta_{t+1} = \theta_t - \text{lr} \cdot v_t$$

### 带权重衰减
$$\theta_{t+1} = \theta_t - \text{lr} \cdot (g_t + \lambda \cdot \theta_t)$$

### Nesterov 动量
先 "看一步" 再调整方向，收敛更快。

In [ ]:
# SGD 的各种配置
model = nn.Linear(3, 1)

# 基本 SGD
sgd_basic = optim.SGD(model.parameters(), lr=0.01)

# 带动量的 SGD（最常用）
sgd_momentum = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 带权重衰减的 SGD
sgd_wd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# 带 Nesterov 动量的 SGD
sgd_nesterov = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

print("SGD 配置示例:")
print(f"  基本 SGD:    lr=0.01")
print(f"  动量 SGD:    lr=0.01, momentum=0.9")
print(f"  权重衰减:    lr=0.01, momentum=0.9, weight_decay=1e-4")
print(f"  Nesterov:    lr=0.01, momentum=0.9, nesterov=True")

In [ ]:
# 手动验证 SGD 动量的更新过程
torch.manual_seed(42)
model = nn.Linear(2, 1, bias=False)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

print("SGD + Momentum 更新过程:")
print(f"初始权重: {model.weight.data}\n")

for step in range(3):
    x = torch.tensor([[1.0, 2.0]])
    target = torch.tensor([[3.0]])
    
    optimizer.zero_grad()
    output = model(x)
    loss = nn.MSELoss()(output, target)
    loss.backward()
    
    print(f"Step {step}:")
    print(f"  loss = {loss.item():.4f}")
    print(f"  grad = {model.weight.grad.data}")
    
    weight_before = model.weight.data.clone()
    optimizer.step()
    
    print(f"  权重: {weight_before} -> {model.weight.data}")
    print()

## 3. Adam（自适应矩估计）

Adam 结合了动量和自适应学习率的优点：

- **一阶矩估计**（均值 $m_t$）：类似动量
- **二阶矩估计**（方差 $v_t$）：自适应调整每个参数的学习率

$$m_t = \beta_1 \cdot m_{t-1} + (1-\beta_1) \cdot g_t$$
$$v_t = \beta_2 \cdot v_{t-1} + (1-\beta_2) \cdot g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
$$\theta_{t+1} = \theta_t - \frac{\text{lr}}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

In [ ]:
# Adam 使用示例
torch.manual_seed(42)
model = nn.Linear(2, 1, bias=False)

# Adam 默认参数：lr=0.001, betas=(0.9, 0.999), eps=1e-8
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Adam 优化过程:")
print(f"初始权重: {model.weight.data}\n")

for step in range(5):
    x = torch.tensor([[1.0, 2.0]])
    target = torch.tensor([[3.0]])
    
    optimizer.zero_grad()
    output = model(x)
    loss = nn.MSELoss()(output, target)
    loss.backward()
    
    optimizer.step()
    print(f"Step {step}: loss={loss.item():.6f}, weight={model.weight.data}")

print("\nAdam 的优势:")
print("  1. 自适应学习率: 每个参数有不同的学习率")
print("  2. 偏差校正: 避免初期估计偏差")
print("  3. 适合稀疏梯度和非平稳目标")

## 4. AdamW（解耦权重衰减）

Adam 的权重衰减实现有问题：它把权重衰减和梯度更新耦合在一起。

**Adam + L2 正则化**（不推荐）：
$$g_t' = g_t + \lambda \cdot \theta_t$$
然后用 $g_t'$ 更新 $m_t$ 和 $v_t$，导致正则化被自适应学习率缩放。

**AdamW（推荐）**：
$$\theta_{t+1} = \theta_t - \text{lr} \cdot \left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \cdot \theta_t\right)$$

权重衰减独立于梯度更新，效果更好。

In [ ]:
# AdamW vs Adam + weight_decay 对比
torch.manual_seed(42)

# 创建两个相同的模型
model_adam = nn.Linear(5, 1)
model_adamw = nn.Linear(5, 1)
model_adamw.load_state_dict(model_adam.state_dict())  # 确保初始权重相同

# Adam with weight_decay（L2 正则化耦合在自适应学习率中）
opt_adam = optim.Adam(model_adam.parameters(), lr=0.01, weight_decay=0.01)

# AdamW（解耦的权重衰减，推荐）
opt_adamw = optim.AdamW(model_adamw.parameters(), lr=0.01, weight_decay=0.01)

# 训练几步
print("Adam vs AdamW 权重变化:")
for step in range(10):
    x = torch.randn(4, 5)
    target = torch.randn(4, 1)
    
    # Adam
    opt_adam.zero_grad()
    loss_adam = nn.MSELoss()(model_adam(x), target)
    loss_adam.backward()
    opt_adam.step()
    
    # AdamW
    opt_adamw.zero_grad()
    loss_adamw = nn.MSELoss()(model_adamw(x), target)
    loss_adamw.backward()
    opt_adamw.step()

print(f"Adam  权重范数: {model_adam.weight.data.norm():.6f}")
print(f"AdamW 权重范数: {model_adamw.weight.data.norm():.6f}")
print("\nAdamW 的权重衰减更有效，权重范数通常更小")
print("\n最佳实践: 使用 AdamW 代替 Adam + weight_decay")

## 5. 参数组（Parameter Groups）

不同层可以使用不同的学习率或其他超参数。

常见场景：
- 预训练 backbone 用小学习率
- 新加的分类头用大学习率
- bias 不使用权重衰减

In [ ]:
# 参数组示例
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(10, 20),
            nn.ReLU(),
            nn.Linear(20, 20),
        )
        self.classifier = nn.Linear(20, 5)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = SimpleNet()

# 不同参数组使用不同学习率
optimizer = optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 1e-4},      # backbone: 小学习率
    {'params': model.classifier.parameters(), 'lr': 1e-3},    # classifier: 大学习率
], weight_decay=1e-5)

print("参数组配置:")
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    print(f"  组 {i}: lr={group['lr']}, weight_decay={group['weight_decay']}, "
          f"参数量={n_params}")

In [ ]:
# 使用 add_param_group 动态添加参数组
model = SimpleNet()

# 先只优化 classifier
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
print(f"初始参数组数: {len(optimizer.param_groups)}")

# 训练几个 epoch 后，添加 backbone 参数
optimizer.add_param_group({'params': model.backbone.parameters(), 'lr': 1e-4})
print(f"添加后参数组数: {len(optimizer.param_groups)}")

for i, group in enumerate(optimizer.param_groups):
    print(f"  组 {i}: lr={group['lr']}")

print("\n应用场景: 分阶段微调（先训练头部，再解冻 backbone）")

In [ ]:
# bias 不使用权重衰减（常见最佳实践）
model = SimpleNet()

# 分离 weight 和 bias 参数
weight_params = []
bias_params = []
for name, param in model.named_parameters():
    if 'bias' in name:
        bias_params.append(param)
    else:
        weight_params.append(param)

optimizer = optim.AdamW([
    {'params': weight_params, 'weight_decay': 0.01},
    {'params': bias_params, 'weight_decay': 0.0},   # bias 不衰减
], lr=1e-3)

print(f"weight 参数: {len(weight_params)} 个")
print(f"bias 参数:   {len(bias_params)} 个")
print("\n最佳实践: bias 和 LayerNorm 的参数通常不使用权重衰减")

## 6. 优化器状态的保存和加载

训练中断后可以从上次的状态继续训练。需要保存：
- 模型参数
- 优化器状态（包括动量、自适应学习率等）

In [ ]:
import os
import tempfile

# 创建模型和优化器
model = nn.Linear(3, 1)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练几步，让优化器有状态
for _ in range(5):
    x = torch.randn(4, 3)
    target = torch.randn(4, 1)
    optimizer.zero_grad()
    loss = nn.MSELoss()(model(x), target)
    loss.backward()
    optimizer.step()

# 查看优化器状态
state = optimizer.state_dict()
print("优化器状态字典的键:")
print(f"  state 键: {list(state['state'].keys())}")
print(f"  param_groups 数量: {len(state['param_groups'])}")

# 查看第一个参数的状态
first_state = state['state'][0]
print(f"\n第一个参数的 Adam 状态:")
print(f"  step: {first_state['step']}")
print(f"  exp_avg (一阶矩): {first_state['exp_avg']}")
print(f"  exp_avg_sq (二阶矩): {first_state['exp_avg_sq']}")

In [ ]:
# 保存和加载完整训练状态
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, 'checkpoint.pt')
    
    # 保存
    torch.save({
        'epoch': 10,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss.item(),
    }, checkpoint_path)
    print(f"保存 checkpoint 到: {checkpoint_path}")
    
    # 加载（模拟重新开始）
    new_model = nn.Linear(3, 1)
    new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)
    
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    new_model.load_state_dict(checkpoint['model_state_dict'])
    new_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    print(f"加载成功!")
    print(f"  恢复到 epoch: {checkpoint['epoch']}")
    print(f"  上次 loss: {checkpoint['loss']:.6f}")
    print(f"  权重一致: {torch.allclose(model.weight.data, new_model.weight.data)}")

## 7. 优化器总结表

PyTorch 提供了 13 种优化器：

| 优化器 | 特点 | 适用场景 |
|--------|------|----------|
| `SGD` | 最基本，需要精细调参 | 图像分类（配合动量和调度器） |
| `SGD+momentum` | 加速收敛 | 大多数 CV 任务 |
| `Adam` | 自适应学习率，收敛快 | NLP、GAN、通用默认选择 |
| `AdamW` | 解耦权重衰减 | Transformer、BERT 微调（推荐） |
| `Adagrad` | 适合稀疏数据 | 推荐系统、NLP 嵌入 |
| `Adadelta` | 无需手动设置学习率 | 通用 |
| `RMSprop` | Adam 的前身 | RNN |
| `Adamax` | Adam 的 L∞ 变体 | 嵌入层参数较大时 |
| `NAdam` | Nesterov + Adam | Adam 的改进 |
| `RAdam` | 自适应的学习率 warmup | 不需要手动 warmup |
| `SparseAdam` | 稀疏梯度 | 大规模嵌入层 |
| `ASGD` | 随机平均 SGD | 凸优化 |
| `LBFGS` | 二阶方法 | 小规模问题、科学计算 |

### 选择建议

```
默认选择 ──> AdamW（大多数任务）
CV 任务 ──> SGD + momentum + scheduler
NLP/Transformer ──> AdamW
微调预训练模型 ──> AdamW + 不同参数组
不确定用啥 ──> 先试 AdamW，再试 SGD+momentum
```

## 8. 练习题

### 练习 1：比较 SGD 和 Adam 的收敛速度

在一个简单的二次函数 $f(x) = (x-3)^2 + (y-5)^2$ 上比较两种优化器的收敛轨迹。

In [ ]:
# 练习 1: 比较 SGD 和 Adam
def optimize_quadratic(optimizer_class, lr, n_steps=100, **kwargs):
    """在二次函数上优化，返回参数轨迹。"""
    # 初始点
    x = torch.tensor([0.0], requires_grad=True)
    y = torch.tensor([0.0], requires_grad=True)
    optimizer = optimizer_class([x, y], lr=lr, **kwargs)
    
    trajectory = []
    for step in range(n_steps):
        optimizer.zero_grad()
        loss = (x - 3) ** 2 + (y - 5) ** 2
        loss.backward()
        optimizer.step()
        trajectory.append((x.item(), y.item(), loss.item()))
    
    return trajectory

# TODO: 用 SGD(lr=0.1) 和 Adam(lr=0.1) 分别优化
# sgd_traj = optimize_quadratic(optim.SGD, lr=0.1)
# adam_traj = optimize_quadratic(optim.Adam, lr=0.1)

# 打印最终结果
# print(f"SGD  最终: x={sgd_traj[-1][0]:.4f}, y={sgd_traj[-1][1]:.4f}, loss={sgd_traj[-1][2]:.6f}")
# print(f"Adam 最终: x={adam_traj[-1][0]:.4f}, y={adam_traj[-1][1]:.4f}, loss={adam_traj[-1][2]:.6f}")

# 打印前 10 步的 loss 对比
# print("\n前 10 步 loss 对比:")
# for i in range(10):
#     print(f"  Step {i}: SGD={sgd_traj[i][2]:.4f}, Adam={adam_traj[i][2]:.4f}")

### 练习 2：实现带参数组的完整训练

创建一个模型，backbone 和 head 使用不同的学习率，完成一个完整的训练循环。

In [ ]:
# 练习 2
# TODO: 创建模型
# TODO: 配置不同参数组的优化器
# TODO: 训练 20 步并打印 loss 变化
pass

## 9. 小结

### 核心要点

1. **5 个核心方法**: `zero_grad`, `step`, `add_param_group`, `state_dict`, `load_state_dict`
2. **SGD**: 基础但强大，配合动量和调度器效果好
3. **Adam**: 自适应学习率，上手快
4. **AdamW**: 解耦权重衰减，推荐替代 Adam
5. **参数组**: 不同层可以有不同的学习率和权重衰减
6. **状态保存**: 训练中断时需要同时保存模型和优化器状态

### 最佳实践

- 默认使用 **AdamW**
- CV 任务可以试试 **SGD + momentum**
- bias 和 normalization 层不使用权重衰减
- 微调时给不同层设置不同学习率
- 始终保存 checkpoint（模型 + 优化器 + epoch）

### 下一步

接下来我们将学习学习率调度器（LR Scheduler），了解如何在训练过程中动态调整学习率。